In [ ]:
## Using dxfs 

## the procedures below half work - the minimum values for each of the dimensions are not being found - defaulting to zero

## TODO fix the above

In [ ]:
# setting up the domain

# Load the DXF file
import ezdxf

# this is a polyline


dxf_file = ezdxf.readfile('/Users/lin236/Documents/CSIRO Share/Export 19032024/MINDOM_A.DXF')

# Get the modelspace
modelspace = dxf_file.modelspace()

# Initialize bounding box variables
min_x, min_y, min_z = float("inf"), float("inf"), float("inf")
max_x, max_y, max_z = float("-inf"), float("-inf"), float("-inf")

# Iterate through all entities in the modelspace
for entity in modelspace:
    if entity.dxftype() == 'LINE':  # Handle LINE entities
        start = entity.dxf.start
        end = entity.dxf.end
        min_x, min_y, min_z = min(min_x, start.x, end.x), min(min_y, start.y, end.y), min(min_z, start.z, end.z)
        max_x, max_y, max_z = max(max_x, start.x, end.x), max(max_y, start.y, end.y), max(max_z, start.z, end.z)
    elif entity.dxftype() == 'POLYLINE':  # Handle POLYLINE entities
        for point in entity.points():
            min_x, min_y, min_z = min(min_x, point[0]), min(min_y, point[1]), min(min_z, point[2])
            max_x, max_y, max_z = max(max_x, point[0]), max(max_y, point[1]), max(max_z, point[2])
    elif entity.dxftype() == 'CIRCLE':  # Handle CIRCLE entities
        center = entity.dxf.center
        radius = entity.dxf.radius
        min_x, min_y, min_z = min(min_x, center.x - radius), min(min_y, center.y - radius), min(min_z, center.z)
        max_x, max_y, max_z = max(max_x, center.x + radius), max(max_y, center.y + radius), max(max_z, center.z)

# Calculate dimensions
east = max_x - min_x
north = max_y - min_y
depth = max_z - min_z

print(f"Bounding Box Dimensions:")
print(f"East max: {max_x}")
print(f"East min: {min_x}")
print(f"North max: {max_y}")
print(f"North min: {min_y}")
print(f"Depth max: {max_z}")
print(f"Depth min: {min_z}")


print(f"East: {east}")
print(f"North: {north}")
print(f"Depth: {depth}")

In [ ]:
import ezdxf
import plotly.graph_objects as go

# Load the DXF file
dxf_file = ezdxf.readfile('/Users/lin236/Documents/CSIRO Share/Export 19032024/MINDOM_A.DXF')

# Get the modelspace
modelspace = dxf_file.modelspace()

# Initialize lists to store coordinates for entities
x_coords = []
y_coords = []
z_coords = []

# Initialize bounding box variables
min_x, min_y, min_z = float("inf"), float("inf"), float("inf")
max_x, max_y, max_z = float("-inf"), float("-inf"), float("-inf")

# Iterate through all entities in the modelspace
for entity in modelspace:
    if entity.dxftype() == 'LINE':  # Handle LINE entities
        start = entity.dxf.start
        end = entity.dxf.end
        x_coords.extend([start.x, end.x, None])  # Add x-coordinates (None for breaks)
        y_coords.extend([start.y, end.y, None])  # Add y-coordinates (None for breaks)
        z_coords.extend([start.z, end.z, None])  # Add z-coordinates (None for breaks)
        # Update bounding box
        min_x, min_y, min_z = min(min_x, start.x, end.x), min(min_y, start.y, end.y), min(min_z, start.z, end.z)
        max_x, max_y, max_z = max(max_x, start.x, end.x), max(max_y, start.y, end.y), max(max_z, start.z, end.z)
    elif entity.dxftype() == 'POLYLINE':  # Handle POLYLINE entities
        for point in entity.points():
            x_coords.append(point[0])
            y_coords.append(point[1])
            z_coords.append(point[2])
            # Update bounding box
            min_x, min_y, min_z = min(min_x, point[0]), min(min_y, point[1]), min(min_z, point[2])
            max_x, max_y, max_z = max(max_x, point[0]), max(max_y, point[1]), max(max_z, point[2])
        x_coords.append(None)  # Add None to break the line
        y_coords.append(None)
        z_coords.append(None)

# Create a 3D scatter plot for the DXF entities
fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=x_coords,
    y=y_coords,
    z=z_coords,
    mode='lines',
    line=dict(color='blue', width=2),
    name='DXF Entities'
))

# Add the bounding box as a wireframe
fig.add_trace(go.Scatter3d(
    x=[min_x, max_x, max_x, min_x, min_x, max_x, max_x, min_x],
    y=[min_y, min_y, max_y, max_y, min_y, min_y, max_y, max_y],
    z=[min_z, min_z, min_z, min_z, max_z, max_z, max_z, max_z],
    mode='lines',
    line=dict(color='red', width=3),
    name='Bounding Box'
))

# Set plot layout
fig.update_layout(
    title="3D Visualization of DXF File and Bounding Box",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
        aspectmode='data'  # Ensures equal scaling for all axes
    )
)

# Show the plot
fig.show()